# Hybrid ARIMA-LSTM Stock Forecasting (corrected)

Forecasts the next day's closing price of **Reliance (RELIANCE.NS)** and the **NIFTY 50 (^NSEI)** index.

Hybrid model (Zhang, 2003): `forecast = ARIMA forecast + LSTM forecast of ARIMA's residuals`.

What this version fixes compared with the first notebook:

1. No look-ahead leakage: every input uses only data up to day t-1.
2. Chronological train / validation / test split. Scalers, ARIMA orders and early stopping use train and validation only. Test is touched once.
3. ARIMA is refreshed one day at a time (walk-forward), without refitting its parameters on test data.
4. Baselines: naive (tomorrow = today), ARIMA only, LSTM only.
5. All metrics are in rupees (index points for NIFTY), not on scaled values.

Split: train 2015-2021, validation 2022, test 2023-2024.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

SEED = 42
START = "2015-01-01"
END = "2025-01-01"
TRAIN_END = "2021-12-31"
VAL_END = "2022-12-31"
LOOKBACK = 60
TICKERS = ["RELIANCE.NS", "^NSEI"]


def set_seed(s=SEED):
    os.environ["PYTHONHASHSEED"] = str(s)
    random.seed(s)
    np.random.seed(s)
    tf.random.set_seed(s)
    tf.config.experimental.enable_op_determinism()

## 1. Data

Downloads adjusted daily closes from Yahoo Finance. If a file `data/<TICKER>.csv` with `Date` and `Close` columns exists, it is used instead (handy offline).

In [ ]:
def load_prices(ticker):
    path = os.path.join("data", ticker.replace("^", "") + ".csv")
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=["Date"])
        s = df.set_index("Date")["Close"]
    else:
        import yfinance as yf
        df = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
        s = df["Close"]
        if isinstance(s, pd.DataFrame):
            s = s.iloc[:, 0]
    s = s.dropna().astype(float)
    s.index = pd.to_datetime(s.index).tz_localize(None)
    s.name = "close"
    return s.sort_index()


prices = {t: load_prices(t) for t in TICKERS}
for t, s in prices.items():
    print(t, len(s), "days,", s.index[0].date(), "to", s.index[-1].date())

## 2. Stationarity (ADF test on the training period only)

H0 of the ADF test: the series has a unit root (non-stationary). p <= 0.05 means stationary.

In [ ]:
def adf(x):
    r = adfuller(x.dropna())
    return r[0], r[1]


rows = []
for t, s in prices.items():
    y = np.log(s[:TRAIN_END])
    a, p = adf(y)
    b, q = adf(y.diff())
    rows.append([t, round(a, 3), round(p, 4), round(b, 3), q])
pd.DataFrame(rows, columns=["ticker", "ADF log price", "p", "ADF log return", "p "])

In [ ]:
for t, s in prices.items():
    r = np.log(s[:TRAIN_END]).diff().dropna()
    fig, ax = plt.subplots(1, 2, figsize=(12, 3))
    plot_acf(r, lags=20, ax=ax[0], title=t + " ACF of log returns")
    plot_pacf(r, lags=20, ax=ax[1], title=t + " PACF of log returns")
    plt.tight_layout()
    plt.show()

## 3. Models

* **ARIMA(p, 1, q)** on log price. Orders chosen by lowest AIC on the training period (p, q from 0 to 3). Fitted once on train; for validation and test the model is updated one day at a time with the new observation, keeping the fitted parameters (no refit on future data).
* **LSTM on residuals**: input = the previous 60 ARIMA residuals, target = the next residual. Scaler fitted on training residuals; early stopping on validation.
* **Hybrid** = ARIMA forecast + LSTM residual forecast.
* **LSTM only** baseline: same LSTM, trained on the previous 60 log returns to predict the next return.
* **Naive** baseline: tomorrow's price = today's price.

In [ ]:
def pick_order(y):
    best = None
    for p in range(4):
        for q in range(4):
            try:
                m = ARIMA(y, order=(p, 1, q)).fit()
            except Exception:
                continue
            if best is None or m.aic < best[0]:
                best = (m.aic, (p, 1, q))
    return best[1]


def arima_walk_forward(y, order):
    tr = y[:TRAIN_END]
    res = ARIMA(tr.values, order=order).fit()
    fit = pd.Series(res.fittedvalues, index=tr.index)
    fc = []
    rest = y[tr.index[-1]:].iloc[1:]
    for v in rest.values:
        fc.append(res.forecast(1)[0])
        res = res.append([v], refit=False)
    fc = pd.Series(fc, index=rest.index)
    pred = pd.concat([fit, fc])
    resid = y - pred
    resid.iloc[:order[1]] = 0.0
    return pred, resid, res


def windows(x, n):
    X, Y = [], []
    for i in range(n, len(x)):
        X.append(x[i - n:i])
        Y.append(x[i])
    return np.array(X)[..., None], np.array(Y)


def build_lstm(n):
    m = Sequential([
        Input(shape=(n, 1)),
        LSTM(64),
        Dropout(0.3),
        Dense(25, activation="relu"),
        Dense(1),
    ])
    m.compile(loss="mse", optimizer="adam")
    return m


def lstm_forecast(series, n=LOOKBACK):
    set_seed()
    tr = series[:TRAIN_END]
    sc = StandardScaler().fit(tr.values.reshape(-1, 1))
    z = pd.Series(sc.transform(series.values.reshape(-1, 1)).ravel(), index=series.index)
    X, Y = windows(z.values, n)
    idx = z.index[n:]
    is_tr = idx <= pd.Timestamp(TRAIN_END)
    is_va = (idx > pd.Timestamp(TRAIN_END)) & (idx <= pd.Timestamp(VAL_END))
    m = build_lstm(n)
    es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    h = m.fit(X[is_tr], Y[is_tr], validation_data=(X[is_va], Y[is_va]),
              epochs=200, batch_size=64, callbacks=[es], verbose=0, shuffle=True)
    out = m.predict(X, verbose=0).ravel()
    out = sc.inverse_transform(out.reshape(-1, 1)).ravel()
    return pd.Series(out, index=idx), len(h.history["loss"])

## 4. Metrics (in price units, on the test period)

* **MAPE**: mean absolute % error.
* **RMSE**: root mean squared error.
* **MASE**: MAE divided by the in-sample MAE of the naive forecast on the training period. Below 1 = better than naive.
* **Direction**: % of days where the predicted move (up/down vs today's close) matches the actual move.

In [ ]:
def metrics(actual, pred, prev, scale):
    e = actual - pred
    mape = np.mean(np.abs(e / actual)) * 100
    rmse = np.sqrt(np.mean(e ** 2))
    mase = np.mean(np.abs(e)) / scale
    move = np.sign(actual - prev)
    d = pred - prev
    d[np.abs(d) < 1e-9 * prev] = 0.0
    guess = np.sign(d)
    ok = move != 0
    if np.all(guess == 0):
        dacc = np.nan
    else:
        dacc = np.mean(move[ok] == guess[ok]) * 100
    return mape, rmse, mase, dacc


def run(ticker):
    s = prices[ticker]
    y = np.log(s)
    order = pick_order(y[:TRAIN_END])
    a_pred, resid, a_res = arima_walk_forward(y, order)
    lb = acorr_ljungbox(resid[:TRAIN_END].iloc[1:], lags=[10], return_df=True)["lb_pvalue"].iloc[0]

    r_pred, r_ep = lstm_forecast(resid)
    ret = y.diff().fillna(0.0)
    l_pred, l_ep = lstm_forecast(ret)

    te = y[VAL_END:].index[1:]
    prev = s.shift(1)
    scale = np.mean(np.abs(s[:TRAIN_END].diff().dropna()))

    f = pd.DataFrame(index=te)
    f["actual"] = s[te]
    f["naive"] = prev[te]
    f["arima"] = np.exp(a_pred[te])
    f["lstm"] = np.exp(y.shift(1)[te] + l_pred[te])
    f["hybrid"] = np.exp(a_pred[te] + r_pred[te])

    rows = []
    for k in ["naive", "arima", "lstm", "hybrid"]:
        rows.append([ticker, k] + list(metrics(f["actual"].values, f[k].values, prev[te].values, scale)))
    out = pd.DataFrame(rows, columns=["ticker", "model", "MAPE %", "RMSE", "MASE", "Direction %"])
    info = {"ticker": ticker, "arima_order": order, "ljung_box_p": lb,
            "lstm_resid_epochs": r_ep, "lstm_only_epochs": l_ep, "test_days": len(te)}
    return out, f, info

## 5. Run both tickers

In [ ]:
results, frames, infos = [], {}, []
for t in TICKERS:
    out, f, info = run(t)
    results.append(out)
    frames[t] = f
    infos.append(info)

table = pd.concat(results, ignore_index=True)
pd.DataFrame(infos)

In [ ]:
table.round(3)

In [ ]:
def gain(t, base):
    x = table[(table.ticker == t)].set_index("model")["MAPE %"]
    return (x[base] - x["hybrid"]) / x[base] * 100


summary = []
for t in TICKERS:
    summary.append([t, gain(t, "naive"), gain(t, "arima"), gain(t, "lstm")])
summary = pd.DataFrame(summary, columns=["ticker", "MAPE cut vs naive %", "vs ARIMA %", "vs LSTM %"])
os.makedirs("results", exist_ok=True)
table.round(4).to_csv("results/metrics.csv", index=False)
summary.round(2).to_csv("results/summary.csv", index=False)
summary.round(2)

Positive numbers mean the hybrid's MAPE is lower than that baseline. Negative means the hybrid is worse.

In [ ]:
for t, f in frames.items():
    plt.figure(figsize=(14, 5))
    plt.plot(f.index, f["actual"], label="Actual", color="black", lw=1)
    plt.plot(f.index, f["hybrid"], label="Hybrid", lw=1)
    plt.plot(f.index, f["arima"], label="ARIMA", lw=1, alpha=0.7)
    plt.title(t + " test period (2023-2024), one-day-ahead forecasts")
    plt.ylabel("Price (INR)" if t != "^NSEI" else "Index points")
    plt.legend()
    plt.tight_layout()
    plt.show()